In [1]:
!pip install ultralytics

   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ----------------------- ---------------- 0.8/1.3 MB 7.8 MB/s eta 0:00:01
   ---------------------------------------- 1.3/1.3 MB 8.7 MB/s  0:00:00
   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
    --------------------------------------- 0.5/40.2 MB 13.8 MB/s eta 0:00:03
   -- ------------------------------------- 2.9/40.2 MB 7.1 MB/s eta 0:00:06
   ----- ---------------------------------- 5.8/40.2 MB 9.5 MB/s eta 0:00:04
   -------- ------------------------------- 8.9/40.2 MB 10.7 MB/s eta 0:00:03
   ----------- ---------------------------- 11.5/40.2 MB 11.3 MB/s eta 0:00:03
   -------------- ------------------------- 14.4/40.2 MB 11.7 MB/s eta 0:00:03
   ----------------- ---------------------- 17.3/40.2 MB 12.0 MB/s eta 0:00:02
   -------------------- ------------------- 20.2/40.2 MB 12.2 MB/s eta 0:00:02
   ---------------------- ----------------- 22.8/40.2 MB 12.4 MB/s eta 0:00:02
   ----

In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from ultralytics import YOLO

# Modelleri yüklüyoruz (İlk çalışmada biraz bekletebilir)
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
yolo_model = YOLO("yolov8n.pt")

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\user\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [7]:
def resim_benzerlik_skoru(orijinal_resim_yolu, yorum_resim_yolu):
    image_a = Image.open(orijinal_resim_yolu)
    image_b = Image.open(yorum_resim_yolu)

    inputs = clip_processor(images=[image_a, image_b], return_tensors="pt", padding=True)
    
    with torch.no_grad():
        outputs = clip_model.get_image_features(**inputs)
        
        if hasattr(outputs, "image_embeds"):
            image_features = outputs.image_embeds
        elif hasattr(outputs, "pooler_output"):
            image_features = outputs.pooler_output
        else:
            image_features = outputs
        
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)
    similarity = torch.clamp(torch.matmul(image_features[0], image_features[1].T), 0, 1)
    
    return similarity.item() * 100

def kargo_kutusu_mu(resim_yolu):
    results = yolo_model(resim_yolu, verbose=False)
    for r in results:
        for box in r.boxes:
            c = box.cls
            etiket_ismi = yolo_model.names[int(c)]
            # Model resimde bavul, çanta veya el çantası gibi kutu benzeri nesneler bulursa
            if etiket_ismi in ["suitcase", "handbag", "backpack"]:
                return True
    return False

In [8]:
def resim_benzerlik_skoru(orijinal_resim_yolu, yorum_resim_yolu):
    image_a = Image.open(orijinal_resim_yolu)
    image_b = Image.open(yorum_resim_yolu)

    # DÜZELTME: 'processor' yerine 'clip_processor' kullanıyoruz
    inputs = clip_processor(images=[image_a, image_b], return_tensors="pt", padding=True)
    
    with torch.no_grad():
        # DÜZELTME: 'model' yerine 'clip_model' kullanıyoruz
        outputs = clip_model.get_image_features(**inputs)
        
        if hasattr(outputs, "image_embeds"):
            image_features = outputs.image_embeds
        elif hasattr(outputs, "pooler_output"):
            image_features = outputs.pooler_output
        else:
            image_features = outputs
        
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)
    
    # Küçük bir PyTorch uyarısını engellemek için .T yerine .mT veya transpose kullanabilirsin
    similarity = torch.clamp(torch.matmul(image_features[0], image_features[1].transpose(0, -1)), 0, 1)
    
    return similarity.item() * 100

def kargo_kutusu_mu(resim_yolu):
    results = yolo_model(resim_yolu, verbose=False)
    for r in results:
        for box in r.boxes:
            c = box.cls
            etiket_ismi = yolo_model.names[int(c)]
            if etiket_ismi in ["suitcase", "handbag", "backpack"]:
                return True
    return False

In [9]:
orijinal = "urun_orijinal.png"
yorum = "kullanici_yorumu.png"

print("1. ADIM: Nesne Kontrolü Yapılıyor...")
if kargo_kutusu_mu(yorum):
    print("Sonuç: REDDEDİLDİ! (Yorum resmi kargo paketi/kutu içeriyor.)")
else:
    print("2. ADIM: Benzerlik Analizi Yapılıyor...")
    skor = resim_benzerlik_skoru(orijinal, yorum)
    print(f"Benzerlik Skoru: %{skor:.2f}")
    
    if skor >= 65.0:
        print("Sonuç: ONAYLANDI! (Görsel ürünle eşleşiyor.)")
    else:
        print("Sonuç: REDDEDİLDİ! (Alakasız görsel.)")

1. ADIM: Nesne Kontrolü Yapılıyor...
2. ADIM: Benzerlik Analizi Yapılıyor...
Benzerlik Skoru: %77.96
Sonuç: ONAYLANDI! (Görsel ürünle eşleşiyor.)
